<h1 style="color:#D4A017; font-weight:900; font-size:34px;">
  🟨 I. RAG Moderne avec LCEL 
</h1>



<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 1 — Utiliser WordIngestion pour préparer les chunks
</h2>


In [ ]:
from word_ingestion import WordIngestion

# ingestor personnalisé 
ingestor = WordIngestion(
    chunk_size= 500,
    chunk_overlap= 80
)

# Traiter un document word 
chunks = ingestor.process("../data/word_files/Document_RAG_Word.docx")

print("Nombre de chunks :", len(chunks))

# Aperçu des premiers chunks 
for i, c in enumerate(chunks[:3]):
    print("=" * 80)
    print(f"CHUNK {i+1}")
    print("=" *80)
    print(c.page_content[:500], "\n")




<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 2 — Créer les embeddings + Vector Store (Chroma)
</h2>


In [ ]:
# Import Embeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Initialisation du Vector Store
vector_store = Chroma(
    collection_name="rag_word_docs",
    persist_directory="chroma_word_store",
    embedding_function=embeddings
)

# Ajouter tous mes chunks dans le vector store
vector_store.add_documents(chunks)

print("Index vectoriel initalisé")

<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 3 — Créer le Retriever
</h2>


In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k" : 3})

In [ ]:
# afficher un exemple de récupération
retriever.invoke("Quels sont les défis liés à l'extraction du texte Word ?")

<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 4 — Définir le Prompt + le pipeline LCEL
</h2>


In [ ]:
# Import
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_template("""
Tu es un assistant expert des systèmes RAG.
Réponds UNIQUEMENT à partir du contexte fourni ci-dessous.
Si l'information n'est pas présente, réponds simplement : "Ce n'est pas précisé dans le document."

Contexte :                                                                                           
{context}

Question :
{question} 
                                                                                                                                                                                                  
Réponse : 
""")

rag_chain = (
    {"context" : lambda x:retriever.invoke(x["question"]),
     "question" : lambda x: x["question"]}
    | prompt
    | llm
    | StrOutputParser()
)

<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 4 — Execution : Poser une question sur le document Word
</h2>


<h3 style="color:#C0392B; font-weight:bold;">
   Ingestion Word -> nettoyage -> chunking -> indexation Chroma -> retriever -> pipeline LCEL


In [ ]:
# 1 ) Définir la question

question = (
    "Quels sont les principaux défis liés à l'extraction de texte "
    "dans un document Word pour un pipeline RAG ?"
)
question

In [ ]:
# 2) Appeler la chaîne LCEL

response = rag_chain.invoke({"question":question})

In [ ]:
# Affichage de la réponse 
print("QUESTION :")
print(question)
print("\nRéponse :")
print(response)


<h1 style="color:#D4A017; font-weight:900; font-size:34px;">
  🟨 II. Ingestion Dynamique
</h1>

In [ ]:
# 1) Etape 1 : Ingestion du nouveau document
from word_ingestion import WordIngestion

# ingestor personnalisé 
ingestor = WordIngestion(
    chunk_size= 500,
    chunk_overlap= 80
)

# Traiter un document word 
new_chunks = ingestor.process("../data/word_files/Document_RAG_Word_v2.docx")

print("Nombre de chunks :", len(chunks))

# Aperçu des premiers chunks 
for i, c in enumerate(chunks[:3]):
    print("=" * 80)
    print(f"CHUNK {i+1}")
    print("=" *80)
    print(c.page_content[:500], "\n")


In [ ]:
# Etape 2 - Enrichir les métadonnées + ajout à Chroma

#1) Ajouter une métadonnée 'source' pour identifier l'origine
for c in new_chunks :
    c.metadata["source"] = "Document_RAG_Word_v2.docx"

#2) Ajouter les nouveaux chunks au vector store existant
vector_store.add_documents(new_chunks)

print("Nouveaux documents ajoutés au vector store")
print(f"Nombre total de documents indexés : {vector_store._collection.count()}")

In [ ]:
# Etape 3 : Vérifier que le retriever "voit" les nouveaux documents

# 1 ) Définir la question

question_new = (
    "Quels sont les principaux défis liés à l'extraction de texte "
    "dans un document Word pour un pipeline RAG ?"
)

# 2) Appeler la chaîne LCEL

response = rag_chain.invoke({"question":question_new})

# Affichage de la réponse 
print("QUESTION :")
print(question)
print("\nRéponse :")
print(response)


<h1 style="color:#D4A017; font-weight:900; font-size:34px;">
  🟨 III. Techniques avancées RAG : Mémoire conversationnelle + Vector Store
</h1>

In [ ]:
# =======================================
#  Etape 1 — IMPORTS & FORMATTEUR DOCS
# =======================================

from operator import itemgetter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough,
    RunnableWithMessageHistory
)
from langchain_core.output_parsers import StrOutputParser

from langchain_community.chat_message_histories import ChatMessageHistory

from pdf_ingestion import PDFIngestion

# Fonction obligatoire pour éviter l'erreur TypeError (liste -> texte)
def format_docs(docs) :
    return "\n\n---\n\n".join(d.page_content for d in docs)

format_docs_runnable = RunnableLambda(format_docs)


In [ ]:
# =======================================
#  Etape 2 — INGESTION PDF 
# =======================================

ingestor = PDFIngestion(chunk_size=1200, chunk_overlap=150) 

chunks_pdf = ingestor.process("../data/pdf_files/rapport-activite_credit_mutuelle.pdf")

print("Nombre de chunks chargés :", len(chunks_pdf))
print("Aperçu du premier chunk :", chunks_pdf[4].page_content[:500])

In [ ]:
# =======================================
#  Etape 3 — INDEXATION DANS CHROMA
# =======================================

embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")

vector_store_pdf = Chroma(
    collection_name="rapport_mutuelle",
    persist_directory= "chroma_pdf_store",
    embedding_function=embeddings
)

# vector_store_pdf.delete_collection()

vector_store_pdf.add_documents(chunks_pdf)

print("Index PDF en base :", vector_store_pdf._collection.count())

In [ ]:
# =======================================
#  Etape 4 — RETRIEVER
# =======================================

retriever_pdf = vector_store_pdf.as_retriever(search_kwargs = {"k":3})
retriever_pdf

In [ ]:
# =======================================
#  Etape 5 — PROMPT AVEC HISTORIQUE
# =======================================

prompt_conv = ChatPromptTemplate.from_messages([
    ("system",
    "Tu es un assistant expert qui analyse le rapport d'activité du Crédit Mutuel."
    "Tu t'appuis uniquement sur les extraits fournis. Réponds clairement et en français"),

    MessagesPlaceholder("history"),

    ("human",
    "Voici les extraits pertinents du rapport : \n\n{context}\n\n"
    "Ma question : {question}")
])


In [ ]:
# =======================================
#  Etape 6 — CHAÎNE RAG LCEL
# =======================================

llm = ChatOpenAI(model = "gpt-4o-mini")

rag_chain = (
    {
        "question" : itemgetter("question"),
        "context" : itemgetter("question") | retriever_pdf | format_docs_runnable,
        "history" : itemgetter("history"),   
    }
    | prompt_conv
    | llm
    | StrOutputParser()
)


In [ ]:
# =======================================
#  Etape 7 — MEMORIRE CONVERSATIONNELLE
# =======================================

store = {}

def get_session_history(session_id : str) : 
    if session_id not in store :
        store[session_id] = ChatMessageHistory()
    return store[session_id]

rag_with_memory = RunnableWithMessageHistory(


    rag_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)



In [ ]:
# =======================================
#  Etape 8 — TEST DU RAG CONVERSATIONNELLE
# =======================================

session_id = "mutuelle_session_1"

def ask(question) :
    response = rag_with_memory.invoke(
        {"question": question},
        config = {"configurable":{"session_id" : session_id}}
    )

    print(f"\n ? {question} \n")
    print("Réponse :\n", response)
    print("\n" + "="*80)

ask("Quels sont les grands axes stratégiques du rapport d'activité ?")
ask("Peux-tu résumer ces axes en mettant l'accent sur la transformation digitale ?")
ask("Comment ces axes stratégiques influencent-ils la relation sociétaire ?")
ask("Et résume moi tout cela en une phrase comme si tu parlais à un sociétaire")

In [ ]:
# =======================================
# 📌 Cellule 8 — TESTS DU RAG
# =======================================

session_id = "mutuelle-session-1"

def ask(question):
    response = rag_with_memory.invoke(
        {"question": question},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"\n❓ {question}\n")
    print("💬 Réponse :\n", response)
    print("\n" + "="*80)

ask("Quels sont les grands axes stratégiques du rapport d'activité ?")
ask("Peux-tu résumer ces axes en mettant l’accent sur la transformation digitale ?")
ask("Comment ces axes stratégiques influencent-ils la relation sociétaire ?")


In [ ]:
ask("Et résume-moi tout cela en une phrase, comme si tu parlais à un sociétaire.")


In [ ]:
ask("Quels sont les principaux défis liés à l'extraction de texte dans un document Word pour un pipeline RAG ?")

In [ ]:
# tester la memoire conversationnelle en posant une question sur un document Word
ask("qui est ISABELLE FERRAND ?")

In [ ]:
ask("Quelle est la position d'ISABELLE FERRAND ?")